# Fig S5B — Heterogeneous effect sizes (data processing)

One locus with 2–3 causal variants of different effect sizes: power to recover the 1st (largest), 2nd and 3rd effect, and FDR, for SuSiE-RSS and SuSiE-rRSS, per LD reference. Bootstrap 95% CIs over replicates.

**Outputs:**
- `figure_data/FigS5B.Heterogeneous_Effects.rds` — plot-ready list(`power_2causal`, `fdr_2causal`, `power_3causal`, `fdr_3causal`)
- `figure_data/FigS5B.Heterogeneous_Effects_summary.{rds,csv}` — full bootstrap summary (also has SuSiE-rRSS -SER, the EB-mix fit without SER fallback)

In [5]:
source("../../R/utils.R")

data_name <- "FigS5B.Heterogeneous_Effects"
n_GWAS    <- 800                      # GTEx eQTL n; NCP = n * h2g

In [6]:
## ---- build SER fallback ------------------------------------------------------
raw <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Hetero_fine/summary/result_susie_minor_effect_summary_summary_minor_effect.rds") %>%
  mutate(h2g = str_extract(input_file, "h2g_[0-9.]+"))

ld_keys   <- c("insample", "ADSP", "UKB", "kg")
keep_cols <- c("found_rank1", "found_rank2", "found_rank3",
               "true_var_number", "true_set_number",
               "total_var_number", "total_set_number", "total_set_length")

## vanilla (SuSiE-RSS) + eb-mix rows
base_tagged <- raw %>%
  filter(method %in% c(ld_keys, paste0(ld_keys, "_mix"))) %>%
  mutate(is_mix = str_detect(method, "_mix$"),
         ldref  = if_else(is_mix, str_remove(method, "_mix$"), method),
         qc     = if_else(is_mix, "SuSiE-rRSS -SER", "SuSiE-RSS")) %>%
  select(input_file, h2g, ldref, qc, all_of(keep_cols))

## SER fallback: reliability flag TRUE -> _mix_SER, else _mix
mix_rows <- raw %>% filter(method %in% paste0(ld_keys, "_mix")) %>%
  mutate(ldref = str_remove(method, "_mix$"))
ser_rows <- raw %>% filter(method %in% paste0(ld_keys, "_mix_SER")) %>%
  mutate(ldref = str_remove(method, "_mix_SER$"))
stopifnot(nrow(mix_rows) > 0, "R_reliability_flag" %in% names(mix_rows))

fallback <- mix_rows %>%
  select(input_file, h2g, ldref, R_reliability_flag, all_of(keep_cols)) %>%
  rename_with(~ paste0(.x, "_mix"), all_of(keep_cols)) %>%
  left_join(ser_rows %>% select(input_file, ldref, all_of(keep_cols)) %>%
              rename_with(~ paste0(.x, "_ser"), all_of(keep_cols)),
            by = c("input_file", "ldref")) %>%
  mutate(use_ser = R_reliability_flag %in% TRUE) %>%
  transmute(
    input_file, h2g, ldref, qc = "SuSiE-rRSS",
    found_rank1      = if_else(use_ser & !is.na(found_rank1_ser),      found_rank1_ser,      found_rank1_mix),
    found_rank2      = if_else(use_ser & !is.na(found_rank2_ser),      found_rank2_ser,      found_rank2_mix),
    found_rank3      = if_else(use_ser & !is.na(found_rank3_ser),      found_rank3_ser,      found_rank3_mix),
    true_var_number  = if_else(use_ser & !is.na(true_var_number_ser),  true_var_number_ser,  true_var_number_mix),
    true_set_number  = if_else(use_ser & !is.na(true_set_number_ser),  true_set_number_ser,  true_set_number_mix),
    total_var_number = if_else(use_ser & !is.na(total_var_number_ser), total_var_number_ser, total_var_number_mix),
    total_set_number = if_else(use_ser & !is.na(total_set_number_ser), total_set_number_ser, total_set_number_mix),
    total_set_length = if_else(use_ser & !is.na(total_set_length_ser), total_set_length_ser, total_set_length_mix)
  )

cell_reps <- bind_rows(base_tagged, fallback) %>% mutate(causal_k = total_var_number)

In [7]:
## ---- per-rank power + FDR, bootstrap 95% CIs (.q() in R/utils.R) -------------
n_boot <- 1000
set.seed(1)
.m <- function(x) { x <- x[is.finite(x)]; if (!length(x)) NA_real_ else mean(x) }

boot_cell_rank <- function(d, B = 1000) {
  R  <- nrow(d)
  pa <- .m(d$found_rank1); ps <- .m(d$found_rank2); pt <- .m(d$found_rank3)
  fdn <- sum(d$total_set_number)
  fdr <- if (fdn == 0) NA_real_ else 1 - sum(d$true_set_number) / fdn
  idx <- matrix(sample.int(R, R * B, replace = TRUE), nrow = R)
  ba <- bs <- bt <- bf <- numeric(B)
  for (b in seq_len(B)) {
    s <- d[idx[, b], ]
    ba[b] <- .m(s$found_rank1); bs[b] <- .m(s$found_rank2); bt[b] <- .m(s$found_rank3)
    den <- sum(s$total_set_number)
    bf[b] <- if (den == 0) NA_real_ else 1 - sum(s$true_set_number) / den
  }
  qa <- .q(ba); qs <- .q(bs); qt <- .q(bt); qf <- .q(bf)
  tibble(R = R,
    power_anchor    = pa, power_anchor_lo    = qa[1], power_anchor_hi    = qa[2],
    power_secondary = ps, power_secondary_lo = qs[1], power_secondary_hi = qs[2],
    power_tertiary  = pt, power_tertiary_lo  = qt[1], power_tertiary_hi  = qt[2],
    FDR             = fdr, FDR_lo            = qf[1], FDR_hi             = qf[2])
}

summary_ci <- cell_reps %>%
  group_by(ldref, qc, h2g, total_var_number) %>%
  group_modify(~ boot_cell_rank(.x, B = n_boot)) %>%
  ungroup()

save_data(summary_ci, paste0(data_name, "_summary"))

In [8]:
## ---- plot-ready: one long table per (number of causal variants, metric set) --
ld_lab    <- c(insample = "In-Sample", ADSP = "ADSP", UKB = "UKB", kg = "1000G")
qc_levels <- c("SuSiE-RSS", "SuSiE-rRSS")       # proposed method last; SuSiE-rRSS -SER stays in the _summary table only

prep <- summary_ci %>%
  filter(qc %in% qc_levels) %>%
  mutate(
    ld  = factor(recode(ldref, !!!ld_lab), levels = unname(ld_lab)),
    qc  = factor(qc, levels = qc_levels),
    h2g_num = as.numeric(str_remove(h2g, "h2g_")),
    ncp_num = n_GWAS * h2g_num,
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "g", digits = 2)), ncp_num)
  )
stopifnot(!any(is.na(prep$qc)), !any(is.na(prep$ld)))

metric_lab <- c(power_anchor = "Power (1st)", power_secondary = "Power (2nd)",
                power_tertiary = "Power (3rd)", FDR = "FDR")

build_long <- function(df, n_causal, ests) {
  d <- df %>% filter(total_var_number == n_causal)
  labs <- metric_lab[ests]
  purrr::map_dfr(ests, function(e)
    d %>% transmute(ld, qc, ncp, metric = labs[[e]],
                    est = .data[[e]], lo = .data[[paste0(e, "_lo")]], hi = .data[[paste0(e, "_hi")]])) %>%
    mutate(metric = factor(metric, levels = unname(labs)))
}

hetero <- list(
  power_2causal = build_long(prep, 2, c("power_anchor", "power_secondary")),
  fdr_2causal   = build_long(prep, 2, "FDR"),
  power_3causal = build_long(prep, 3, c("power_anchor", "power_secondary", "power_tertiary")),
  fdr_3causal   = build_long(prep, 3, "FDR")
)
save_data(hetero, data_name)